In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import logging

import matplotlib.pyplot as plt  # type: ignore
import numpy as np
import healpy as hp

from mlpng import Core, get_itotcov_ell
from mlpng.utils import (
    setup_logging,
    plot_cl_alm,
    plot_patches,
    plot_predictions,
    plot_histogram,
    plot_elsner_comp,
    pol_str,
)
from mlpng.generator import generate_alm, generate_alm_ng, patch_and_lens
from mlpng.utils.utils import print_errors

logger = setup_logging(__name__, level=logging.DEBUG)

## Setup

In [3]:
core = Core(
    [
        "settings/planck.json",
        "--nsims",
        "1",
        "--narray",
        "1",
        "--pols",
        "T",
        # "--lmax",
        # "1000",
    ]
)

29-Oct-24 09:20:40 - mlpng.core - INFO - Parsing CLI args: ['settings/planck.json', '--nsims', '1', '--narray', '1', '--pols', 'T']
29-Oct-24 09:20:40 - mlpng.core - INFO - Loading settings from file 'settings/planck.json'
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Forcing setting 'nsims' to 1 due to CLI
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Forcing setting 'narray' to 1 due to CLI
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Forcing setting 'pols' to ['T'] due to CLI
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 67.66, 'As': 2.105e-09, 'ns': 0.9665, 'ombh2': 0.02242, 'omch2': 0.11933, 'tau': 0.0561} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05})
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Overriding cosmo param 'As' from 2.13e-09 to 2.105e-09
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Overriding cosmo param 'ns' from 0.9624 to 0.9665


29-Oct-24 09:20:40 - mlpng.core - INFO - Running with settings: 
{
  "cosmo_params": {
    "As": 2.105e-09,
    "ns": 0.9665,
    "pivot_scalar": 0.05,
    "H0": 67.66,
    "ombh2": 0.02242,
    "omch2": 0.11933,
    "tau": 0.0561
  },
  "nsims": 1,
  "narray": 1,
  "npatches": 10,
  "nside": 2048,
  "lmax": 2000,
  "lensing": false,
  "noise": true,
  "noise_tt": 29.336,
  "beam_width": 5,
  "base_name": "planck",
  "pols": [
    "T"
  ]
}
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Setting 'seed' not found, using default: 2462527842
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Found non-default value for 'nside': 2048 (default: 1024)
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Found default value for 'lensing': False
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Found non-default value for 'nsims': 1 (default: 100)
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Found non-default value for 'narray': 1 (default: 100)
29-Oct-24 09:20:40 - mlpng.core - DEBUG - Setting 'force_generation' not found, using

In [4]:
core.init_estimator(verbose=False)

theta_batch = int(np.floor(1.5 * core.lmax + 1)) // core.n_cpus

In [5]:
def convert(x):
    # convert arcmin to (rad)^2
    return (x * np.pi / 180.0 / 60.0) ** 2


fwhm_acrmin = np.array([9.2, 7.1, 5.0])
fwhm = np.array([convert(x) for x in fwhm_acrmin])

delta_t = np.array([51.0, 43.0, 65.0])
delta_t_2 = np.array([convert(x) for x in delta_t])[:, None]

sigma = (fwhm / 8.0 / np.log(2.0))[:, None]
ell = np.arange(core.nell)
g = np.exp(ell * (ell + 1) * sigma)

nell_mu = delta_t_2 * g

b_ell = np.array([1 / np.sum(1 / g, axis=0)])
ib = 1 / b_ell
noise = 1 / np.sum(1 / nell_mu, axis=0)

noise = np.atleast_2d(noise)
inoise = 1 / (noise)
inoise = inoise[None, ...]

s_ell = core.c_ell[core.pol_idxs()] + noise
ic = 1 / (s_ell)  # + ib + noise + ib)

inoise[..., : core.lmin] = 1e-16
inoise[inoise == np.inf] = 1e16

ic[..., : core.lmin] = 0
ic[ic == np.inf] = 1e16

icov = get_itotcov_ell(ic, inoise)  # , b_ell)

fisher = core.estimator.compute_fisher_isotropic(icov, fsky=0.8)
print(f"Fisher: {fisher}, Fisher error: {1 / np.sqrt(fisher)}")

Fisher: 0.026132574304938316, Fisher error: 6.185985716777164


In [6]:
s_ell = core.s_ell  # + noise
inoise = 1 / core.nb_ell
inoise[..., : core.lmin] = 1e-16
ib = b_ell

icov = 1 / (s_ell)# + ib * noise * ib)
icov[..., : core.lmin] = 0
icov = get_itotcov_ell(icov, inoise)

fisher = core.estimator.compute_fisher_isotropic(icov, fsky=0.8)
print(f"Fisher: {fisher}, Fisher error: {1 / np.sqrt(fisher)}")
# expecting fisher error of 6.3 for local, smith and zal table 4

/tmp/ipykernel_3566044/4001277842.py:6: RuntimeWarning: divide by zero encountered in divide
  icov = 1 / (s_ell)# + ib * noise * ib)


Fisher: 0.03966912999749184, Fisher error: 5.020808464847798


In [7]:
stop

NameError: name 'stop' is not defined

In [ ]:
alm_l = generate_alm(core)
alm_ng = generate_alm_ng(core, alm_l)

fnls = core.rng.uniform(core.fnl_min, core.fnl_max, (core.nsims,))

alms = alm_l + fnls[:, None, None] * alm_ng

### Plots

In [ ]:
plot_cl_alm(
    core,
    alm_l[0],
    title="linear alms",
    # labels=plt_labels,
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=True,
    show=True,
)

plot_cl_alm(core, alm_ng[0], title="ng alms", show=True)

In [ ]:
plot_cl_alm(
    core,
    alms[0],
    title="alms final",
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=True,
    show=True,
)

## Lensing and Patching

In [ ]:
patches, alm_lensed = patch_and_lens(core, alms)

### Plots

In [ ]:
for pol in range(core.npols):
    plot_patches(patches[0, :, pol], title=f"Pol {pol}", show=True)

In [ ]:
sim = 0

for i, pol in enumerate(core.pol_idxs()):
    pstr = pol_str(pol)
    plot_patches(
        patches[sim, :, i], title=f"Patches for sim: {sim}, pol: {pstr}", show=True
    )

    ylabel = r"$\ell(\ell+1)/2\pi\;C_{\ell}" + f"^{pstr}$"
    plot_cl_alm(
        core,
        alms[sim, i],
        # save_file=filebase,
        plot_camb=True,
        ylabel=ylabel,
        plot_noise=False,
        plot_full_camb=True,
        show=True,
    )

In [ ]:
sim = core.rng.integers(core.nsims)
eidx = core.rng.integers(1, 1001)
plot_elsner_comp(
    core,
    alm_l[sim],
    alm_ng[sim],
    index=eidx,
    show=True,
    elsner_pols=core.pol_idxs(),
)
plot_elsner_comp(
    core,
    alm_l[sim],
    alm_ng[sim],
    index=eidx,
    show=True,
    plot_func=plt.plot,
    elsner_pols=core.pol_idxs(),
)
plot_elsner_comp(
    core,
    alm_l[sim],
    alm_ng[sim],
    index=eidx,
    show=True,
    elsner_pols=core.pol_idxs(),
    plot_func=plt.loglog,
)

In [ ]:
alm_l_avg = np.mean(alm_l, axis=0)
alm_ng_avg = np.mean(alm_ng, axis=0)
plot_elsner_comp(
    core,
    alm_l_avg,
    alm_ng_avg,
    average=10,
    title="avged comp",
    show=True,
    plot_func=plt.semilogy,
    elsner_pols=core.pol_idxs(),
)

## Estimator

In [ ]:
def alm_loader(i):
    if core.lensing:
        return alm_lensed[i]
    else:
        return alms[i]


def icov_func(alm):
    ret = np.zeros_like(alm)
    for pol in range(np.shape(alm)[0]):
        # we drop TE
        ret[pol] = hp.almxfl(alm[pol], icov[pol])
    return ret


idxs = range(alms.shape[0])
estimates, _, _, _ = core.estimator.compute_estimate_batch(
    lambda i: icov_func(alm_loader(i)),
    idxs,
    theta_batch=theta_batch,
    fisher=fisher,
    lin_term=0,
)

In [ ]:
print_errors(fnls, estimates, fisher)
plot_predictions(fnls, estimates, fisher=fisher, show=True)
plot_histogram(fnls, estimates, show=True)